# SqueezeNet 1.1 on EuroSAT RGB

Train and evaluate SqueezeNet using the shared `eurosat_splits.csv` and
`eurosat_classes.json`. This notebook preserves the existing split assignments,
64 x 64 preprocessing, two training stages, and result exports.

Install `requirements.txt`, open this notebook with a Python kernel in the
repository folder, edit the configuration below, and run the cells in order.
The first run downloads ImageNet weights; leaving `data_dir=None` also downloads
the EuroSAT dataset through KaggleHub. Downloads require internet access.
Training uses CUDA when available, otherwise the CPU.

Restart the kernel and run all cells for a fresh experiment. Re-running a training
cell continues with the current in-memory model and history. Outputs are saved
to `output_dir`; choose a new directory to keep previous experiment results.


## Configuration

Set `data_dir` to `Path("C:/datasets/EuroSAT")` if the images are already available;
that directory must contain the ten class folders. Paths below are relative to
the repository folder. Set `ROOT` explicitly if your kernel starts elsewhere.

The defaults train the classifier for 5 epochs and fine-tune for 20. For a shorter
trial, set both epoch counts to 1; this still uses every image in the manifest.
`num_workers=0` supports dataset classes defined in a notebook, including on
Windows. `show_plots=True` displays charts as well as saving them.


In [ ]:
from pathlib import Path
from types import SimpleNamespace

ROOT = Path.cwd().resolve()
if not (ROOT / "eurosat_data.py").is_file():
    raise FileNotFoundError("Set ROOT to the repository folder containing eurosat_data.py")

# Keep the shared helper importable if ROOT is set explicitly above.
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

args = SimpleNamespace(
    data_dir=None,  # Or Path("C:/datasets/EuroSAT")
    splits=ROOT / "eurosat_splits.csv",
    classes=ROOT / "eurosat_classes.json",
    output_dir=ROOT / "squeezenet_results",
    seed=42,
    batch_size=64,
    num_workers=0,
    head_epochs=5,
    fine_tune_epochs=20,
    show_plots=True,
)

for name in ("batch_size", "head_epochs", "fine_tune_epochs"):
    value = getattr(args, name)
    if not isinstance(value, int) or isinstance(value, bool) or value < 1:
        raise ValueError(f"{name} must be a positive integer")
if not isinstance(args.num_workers, int) or isinstance(args.num_workers, bool) or args.num_workers < 0:
    raise ValueError("num_workers must be a nonnegative integer")


## Imports and reproducibility

Seed the random generators, select the device, and create the output folder.


In [ ]:
import json
import time
import random

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
if not args.show_plots:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import (
    squeezenet1_1,
    SqueezeNet1_1_Weights
)

from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


SEED = args.seed

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUTPUT_DIR = args.output_dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", device)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)


## Dataset adapter

Open each manifest image as RGB and apply the shared preprocessing.


In [ ]:
class EuroSATDataset:
    def __init__(self, manifest, split, root, transform):
        self.frame = (
            manifest[manifest["split"] == split]
            .reset_index(drop=True)
        )
        self.root = Path(root)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        from PIL import Image

        row = self.frame.iloc[index]

        with Image.open(
            self.root / row["relative_path"]
        ) as image:
            image = image.convert("RGB")
            image = self.transform(image)

        label = int(row["label"])
        return image, label


## Load the shared splits and images

Validate the supplied manifest and class order, then check that all images exist.


In [ ]:
from eurosat_data import load_splits, validate_images

rows, class_names = load_splits(args.splits, args.classes)
manifest = pd.DataFrame(rows)
NUM_CLASSES = len(class_names)
print(manifest["split"].value_counts())
print("Classes:", class_names)


if args.data_dir is None:
    import kagglehub
    dataset_path = kagglehub.dataset_download("apollo2506/eurosat-dataset")
    DATA_DIR = Path(dataset_path) / "EuroSAT"
else:
    DATA_DIR = args.data_dir

validate_images(rows, DATA_DIR)
print("Dataset directory:", DATA_DIR)
print("All manifest images found.")


## Preprocessing and data loaders

Resize to 64 x 64 and normalize with ImageNet statistics, without augmentation. Shuffle only training batches.


In [ ]:
IMAGE_SIZE = (64, 64)
BATCH_SIZE = args.batch_size

# Preserve the notebook's preprocessing without augmentation.
image_transform = transforms.Compose([
    transforms.Resize(
        IMAGE_SIZE,
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


train_dataset = EuroSATDataset(
    manifest, "train", DATA_DIR, image_transform
)
val_dataset = EuroSATDataset(
    manifest, "val", DATA_DIR, image_transform
)
test_dataset = EuroSATDataset(
    manifest, "test", DATA_DIR, image_transform
)

generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    num_workers=args.num_workers,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=args.num_workers,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=args.num_workers,
    pin_memory=(device.type == "cuda")
)

images, labels = next(iter(train_loader))

print("Image batch:", images.shape)
print("Label batch:", labels.shape)


## Prepare SqueezeNet

Load pretrained weights, freeze the feature extractor, and replace the classifier with one logit per class.


In [ ]:
weights = SqueezeNet1_1_Weights.DEFAULT

model = squeezenet1_1(weights=weights)

# Freeze the pretrained feature extractor initially.
for parameter in model.features.parameters():
    parameter.requires_grad = False

# A new classifier produces one logit per EuroSAT class.
# No softmax: CrossEntropyLoss works directly with logits.
model.classifier = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Conv2d(
        in_channels=512,
        out_channels=NUM_CLASSES,
        kernel_size=1
    ),
    nn.AdaptiveAvgPool2d((1, 1))
)

nn.init.normal_(
    model.classifier[1].weight,
    mean=0.0,
    std=0.01
)
nn.init.zeros_(model.classifier[1].bias)

model.num_classes = NUM_CLASSES
model = model.to(device)


def parameter_counts(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )
    return total, trainable


total, trainable = parameter_counts(model)

print(f"Total parameters: {total:,}")
print(f"Initially trainable parameters: {trainable:,}")

with torch.no_grad():
    output = model(images[:2].to(device))

assert output.shape == (2, NUM_CLASSES)
print("Output shape:", output.shape)
